## Load libraries

In [ ]:
%matplotlib inline


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
%load_ext IPython.extensions.autoreload
%autoreload 2

import hypyp.fnirs as fnirs
from hypyp.wavelet import ComplexMorletWavelet, ComplexGaussianWavelet
from hypyp.utils import Task

## Load raw data

We use the FCS01 fNIRS recording (parent participant) from HypypData.
`datasets.fnirs_fcs01_parent()` downloads all required Homer2/NIRX companion files
into `~/mne_data/HypypData/` and returns the path to the `.hdr` entry point.


In [ ]:
from hypyp import datasets

my_file = datasets.fnirs_fcs01_parent()


## Lag comparison for different wavelets

We load a single subject file and set a single task, but starting with a lag, to create "dyads". We then compare the coherence for these lags, given a set of wavelet configurations


In [ ]:
import os

preprocessor = fnirs.MnePreprocessorRawToHaemo()

recordings = []
dyads = []

recordings.append(fnirs.Recording(subject_label='base', tasks=[Task('baseline', onset_time=0, duration=100)]))
recordings.append(fnirs.Recording(subject_label='1sec', tasks=[Task('baseline', onset_time=1, duration=101)]))
recordings.append(fnirs.Recording(subject_label='3sec', tasks=[Task('baseline', onset_time=3, duration=103)]))
recordings.append(fnirs.Recording(subject_label='5sec', tasks=[Task('baseline', onset_time=5, duration=105)]))
recordings.append(fnirs.Recording(subject_label='10sec', tasks=[Task('baseline', onset_time=10, duration=110)]))

for subject in recordings:
    subject.load_file(my_file, preprocessor)

for recording_with_lag in recordings[1:]:
    dyads.append(fnirs.Dyad(recordings[0], recording_with_lag, label=recording_with_lag.subject_label))

study = fnirs.Study(dyads)

wavelets = [
    ComplexMorletWavelet(period_range=(2, 100)),
    ComplexGaussianWavelet(degree=1, period_range=(2, 100)),
    ComplexGaussianWavelet(degree=2, period_range=(2, 100)),
    ComplexGaussianWavelet(degree=3, period_range=(2, 100)),
]

mosaic = [
    ['wavelet', 'wavelet'],
    ['d0', 'd1'],
    ['d2', 'd3'],
]

for wavelet in wavelets:
    study.compute_wtcs(ch_match=('S1_D1 hbo', 'S1_D2 hbo'), wavelet=wavelet)

    fig, ax_dict = plt.subplot_mosaic(mosaic, figsize=(16, 16))
    fig.suptitle(f'wavelet name: {wavelet.wavelet_name_with_args}')
    wavelet.plot_mother_wavelet(ax=ax_dict['wavelet'])
    for i, dyad in enumerate(study.dyads):
        _ = dyad.wtcs[0].plot(title=dyad.label, ax=ax_dict[f'd{i}'])
